## Downscaling and moving windows
- Downscaling building volume and tree volume rasters to 30m to have same-level scale with LST images 
- computing moving windows and study the correlations with LST depending on the size of the window.

### Libraries and general paths

In [ ]:
!pip install rasterio

In [ ]:
import matplotlib.pyplot as plt
import rasterio
from rasterio.mask import mask
import numpy as np
from rasterio.windows import Window
from rasterio.warp import reproject, Resampling
from scipy.ndimage import uniform_filter
import os
from scipy.ndimage import convolve

In [ ]:
lst_cropped_path = '../../../datos-notebooks/0.LST/LST_cropped_celsius.tif'
lst_clipped_path = '../../../datos-notebooks/0.LST/LST_clipped_celsius.tif'

# Original rasters with height values at 0.5m resolution
trees_raster_path = '../../../datos-notebooks/1.HR-data-bologna/1.2.objects-height/rasterized-0.5m/trees_height_0.5m.tif'
building_raster_path = '../../../datos-notebooks/1.HR-data-bologna/1.2.objects-height/rasterized-0.5m/buildings_height_0.5m.tif'
build_footprint = '../../../datos-notebooks/1.HR-data-bologna/1.2.objects-height/rasterized-0.5m/buildings_footprint_0.5m.tif' # derived from building_raster_path

# Downscaled rasters at 30m resolution
downscaled_folder = '../../../datos-notebooks/1.HR-data-bologna/1.2.objects-height/downscaled/'
os.makedirs(downscaled_folder, exist_ok=True)
build_footprint_30m = os.path.join(downscaled_folder, 'buildings_footprint_30m.tif')
build_volume_30m = os.path.join(downscaled_folder, 'buildings_volume_30m.tif')
tree_volume_30m = os.path.join(downscaled_folder, 'trees_volume_30m.tif')
build_height_30m = os.path.join(downscaled_folder, 'buildings_height_30m.tif')

# Moving windows 
mw_folder = '../../../datos-notebooks/1.HR-data-bologna/1.2.objects-height/moving-windows/'
os.makedirs(mw_folder, exist_ok=True)
mw_building_area = os.path.join(mw_folder, 'building-area/')
os.makedirs(mw_building_area, exist_ok=True)
mw_building_volume = os.path.join(mw_folder, 'building-volume/')
os.makedirs(mw_building_volume, exist_ok=True)
mw_tree_volume = os.path.join(mw_folder, 'tree-volume/')
os.makedirs(mw_tree_volume, exist_ok=True)
mw_building_height = os.path.join(mw_folder, 'building-height/')
os.makedirs(mw_building_height, exist_ok=True)

### Downscaling

#### Building Height

In [ ]:
with rasterio.open(building_raster_path) as src:
    window = rasterio.windows.Window(col_off=15000, row_off=7000, width=3000, height=3000)  # adjust as needed
    data = src.read(1, window=window)

plt.figure(figsize=(8,8))
plt.imshow(data, cmap='viridis')
plt.colorbar(label='Height')
plt.show()

In [ ]:
from rasterio.warp import reproject, Resampling
import numpy as np
import rasterio

# Open LST for spatial reference
with rasterio.open(lst_cropped_path) as src_lst:
    lst_profile = src_lst.profile
    lst_transform = src_lst.transform
    lst_crs = src_lst.crs
    lst_shape = (src_lst.height, src_lst.width)

# Read building height raster
with rasterio.open(building_raster_path) as src:
    height_data = src.read(1).astype(float)
    height_transform = src.transform
    height_crs = src.crs

# Mask zeros (ignore them in average)
height_data_masked = np.where(height_data > 0, height_data, np.nan)

# Prepare output array
resampled = np.empty(lst_shape, dtype=np.float32)

# Reproject using spatial average (ignores NaNs)
reproject(
    source=height_data_masked,
    destination=resampled,
    src_transform=height_transform,
    src_crs=height_crs,
    dst_transform=lst_transform,
    dst_crs=lst_crs,
    resampling=Resampling.average,
    src_nodata=np.nan,
    dst_nodata=np.nan
)

# Replace any remaining NaN with 0
resampled = np.nan_to_num(resampled, nan=0.0)

# Save result
out_profile = lst_profile.copy()
out_profile.update(
    dtype=rasterio.float32,
    count=1,
    compress='lzw'
)

with rasterio.open(build_height_30m, 'w', **out_profile) as dst:
    dst.write(resampled, 1)

print("Saved:", build_height_30m)



In [ ]:
with rasterio.open(build_height_30m) as src:
    build_footprint_data = src.read(1)

plt.figure(figsize=(8,6))
plt.imshow(build_footprint_data, cmap='viridis')
plt.colorbar(label='Built height (m)')
plt.title('Building (mean) height at 30m resolution')
plt.show()

#### Building area

In [ ]:
with rasterio.open(building_raster_path) as src:
    data = src.read(1)
    profile = src.profile

# Create binary raster: 1 = built area, 0 = no building
footprint = np.where(np.isnan(data) | (data <= 0), 0, 1).astype(np.uint8)

# Save raster
profile.update(dtype=rasterio.uint8, count=1, compress='lzw')

with rasterio.open(build_footprint, 'w', **profile) as dst:
    dst.write(footprint, 1)

print("Saved:", build_footprint)


In [ ]:
with rasterio.open(build_footprint) as src:
    window = rasterio.windows.Window(col_off=15000, row_off=7000, width=8000, height=8000)  # adjust as needed
    data = src.read(1, window=window)

plt.figure(figsize=(8,8))
plt.imshow(data, cmap='viridis')
plt.colorbar(label='Footprint')
plt.show()

In [ ]:
# Open LST for spatial reference
with rasterio.open(lst_cropped_path) as src_lst:
    lst_profile = src_lst.profile
    lst_transform = src_lst.transform
    lst_crs = src_lst.crs
    lst_shape = (src_lst.height, src_lst.width)
    lst_res = src_lst.res[0]  # pixel size (e.g. 30 m)

# Read building footprint raster
with rasterio.open(build_footprint) as src:
    footprint_data = src.read(1).astype(float)
    footprint_transform = src.transform
    footprint_crs = src.crs
    footprint_res = src.res[0]  # pixel size (e.g. 0.5 m)

# Reproject with sum (to count built subpixels per LST pixel)
resampled = np.empty(lst_shape, dtype=np.float32)

reproject(
    source=footprint_data,
    destination=resampled,
    src_transform=footprint_transform,
    src_crs=footprint_crs,
    dst_transform=lst_transform,
    dst_crs=lst_crs,
    resampling=Resampling.sum
)

# Convert to built fraction
scale_factor = (lst_res / footprint_res) ** 2  # number of 0.5m pixels per LST pixel
fraction_built = resampled / scale_factor      # fraction between 0 and 1

# Save result
out_profile = lst_profile.copy()
out_profile.update(dtype=rasterio.float32, count=1, compress='lzw')

with rasterio.open(build_footprint_30m, 'w', **out_profile) as dst:
    dst.write(fraction_built, 1)

print("Saved:", build_footprint_30m) 


In [ ]:
with rasterio.open(build_footprint_30m) as src:
    build_footprint_data = src.read(1)

plt.figure(figsize=(8,6))
plt.imshow(build_footprint_data, cmap='viridis')
plt.colorbar(label='Built fraction')
plt.title('Built fraction at 30m resolution')
plt.show()

#### Building volume

In [ ]:
# Load LST to get shape, transform, CRS, and pixel size
with rasterio.open(lst_cropped_path) as src_lst:
    lst_crs = src_lst.crs
    lst_transform = src_lst.transform
    lst_height = src_lst.height
    lst_width = src_lst.width
    lst_res_x = src_lst.res[0]  # Size of LST pixel in X
    lst_res_y = src_lst.res[1]  # Size of LST pixel in Y

# Open 0.5m building raster
with rasterio.open(build_raster_path) as src_build:
    build_data = src_build.read(1)
    build_res = src_build.res[0]  
    build_transform = src_build.transform

# Aggregation factor
factor_x = int(lst_res_x / build_res)
factor_y = int(lst_res_y / build_res)
    
# Initialize volume raster
volume_raster = np.zeros((lst_height, lst_width), dtype=np.float32)
pixel_area = build_res * build_res  # area of each 0.5 m pixel

# Downscaling by summing volumes
for i in range(lst_height):
    for j in range(lst_width):
        # calculate window of pixels in build_data that fall within the LST pixel
        row_start = i * factor_y
        row_end = min((i+1) * factor_y, build_data.shape[0])
        col_start = j * factor_x
        col_end = min((j+1) * factor_x, build_data.shape[1])

        # extract block and sum volumes
        block = build_data[row_start:row_end, col_start:col_end]
        block = np.nan_to_num(block)  # convert NaN to 0
        volume_raster[i, j] = np.sum(block * pixel_area)

# Save raster
with rasterio.open(
    build_volume_30m,
    'w',
    driver='GTiff',
    height=lst_height,
    width=lst_width,
    count=1,
    dtype=volume_raster.dtype,
    crs=lst_crs,
    transform=lst_transform
) as dst:
    dst.write(volume_raster, 1)


In [ ]:
plt.figure(figsize=(8,6))
plt.imshow(volume_raster, cmap='viridis', vmin=0, vmax=15000)  # values up to 50.000 but vmax set for better visualization
plt.colorbar(label='Built volume (m³)')
plt.title('Built volume')
plt.show()

#### Trees volume

In [ ]:
# Load LST to get shape, transform, CRS, and pixel size
with rasterio.open(lst_cropped_path) as src_lst:
    lst_crs = src_lst.crs
    lst_transform = src_lst.transform
    lst_height = src_lst.height
    lst_width = src_lst.width
    lst_res_x = src_lst.res[0]  # tamaño de pixel LST en X
    lst_res_y = src_lst.res[1]  # tamaño de pixel LST en Y

# Load high-res tree raster
with rasterio.open(trees_raster_path) as src_tree:
    build_data = src_tree.read(1)
    build_res = src_tree.res[0]  # asumimos cuadrado 0.5m
    build_transform = src_tree.transform

#Aggregation factor
factor_x = int(lst_res_x / build_res)
factor_y = int(lst_res_y / build_res)

# Inicialyze volume raster
volume_raster = np.zeros((lst_height, lst_width), dtype=np.float32)
pixel_area = build_res * build_res  # área de cada pixel 0.5 m

# Downscaling by summing volumes
for i in range(lst_height):
    for j in range(lst_width):
 
        row_start = i * factor_y
        row_end = min((i+1) * factor_y, build_data.shape[0])
        col_start = j * factor_x
        col_end = min((j+1) * factor_x, build_data.shape[1])
        
        # extract block and sum volumes
        block = build_data[row_start:row_end, col_start:col_end]
        block = np.nan_to_num(block)  # convertir NaN a 0
        volume_raster[i, j] = np.sum(block * pixel_area)

# Save raster
with rasterio.open(
    tree_volume_30m,
    'w',
    driver='GTiff',
    height=lst_height,
    width=lst_width,
    count=1,
    dtype=volume_raster.dtype,
    crs=lst_crs,
    transform=lst_transform
) as dst:
    dst.write(volume_raster, 1)

In [ ]:
plt.figure(figsize=(8,6))
plt.imshow(volume_raster, cmap='viridis', vmin=0, vmax=10000)  # values up to 50.000 but vmax set for better visualization
plt.colorbar(label='vegetation volume (m³)')
plt.title('vegetation volume')
plt.show()

### moving windows

In [ ]:
def distance_weighted_kernel(size):
    """Create a distance-weighted Gaussian kernel."""
    assert size % 2 == 1, "Window size must be odd"
    center = size // 2
    y, x = np.ogrid[:size, :size]
    distance = np.sqrt((x - center)**2 + (y - center)**2)
    sigma = size / 4  # controls smoothing strength
    weights = np.exp(-(distance**2) / (2 * sigma**2))
    weights /= weights.sum()  # normalize to sum=1
    return weights


def moving_window_weighted(array, kernel):
    """Apply distance-weighted convolution."""
    return convolve(array, kernel, mode='nearest')

#### Building area

In [ ]:
with rasterio.open(build_footprint_30m) as src:
    data = src.read(1).astype(np.float32)
    profile = src.profile

    # Loop over odd window sizes only
    for window_size in range(1, 101, 2):
        kernel = distance_weighted_kernel(window_size)
        smoothed = moving_window_weighted(data, kernel)

        # Save raster
        out_profile = profile.copy()
        out_profile.update(dtype=rasterio.float32, count=1, compress='lzw')
        out_path = os.path.join(
            mw_building_area, 
            f"building_area_window{window_size}.tif"
        )

        with rasterio.open(out_path, "w", **out_profile) as dst:
            dst.write(smoothed.astype(np.float32), 1)


In [ ]:
# plot

window_sizes = list(range(1, 21, 2))  # odd windows only 

fig, axes = plt.subplots(2, 5, figsize=(20, 8)) 
axes = axes.flatten()

for ax, w in zip(axes, window_sizes):
    raster_path = os.path.join(mw_building_area, f"building_area_window{w}.tif")
    with rasterio.open(raster_path) as src:
        data = src.read(1)  # read first band directly

    im = ax.imshow(data, cmap='viridis')
    ax.set_title(f"Window {w}x{w}")
    ax.axis("off")

fig.suptitle("Building area - Distance-weighted moving window", fontsize=16)
plt.tight_layout()
plt.show()

#### Building volume

In [ ]:
with rasterio.open(build_volume_30m) as src:
    data = src.read(1).astype(np.float32)
    profile = src.profile

    # Loop over odd window sizes only
    for window_size in range(1, 101, 2):
        if window_size == 1:
            smoothed = data.copy()  # window 1 = original
        else:
            kernel = distance_weighted_kernel(window_size)
            smoothed = moving_window_weighted(data, kernel)

        # Save raster
        out_profile = profile.copy()
        out_profile.update(dtype=rasterio.float32, count=1, compress='lzw')
        out_path = os.path.join(
            mw_building_volume, 
            f"buildings_window{window_size}.tif"
        )

        with rasterio.open(out_path, "w", **out_profile) as dst:
            dst.write(smoothed.astype(np.float32), 1)


In [ ]:
window_sizes = list(range(1, 21, 2))  # odd windows only 

fig, axes = plt.subplots(2, 5, figsize=(20, 8))  # 10 windows
axes = axes.flatten()

for ax, w in zip(axes, window_sizes):
    raster_path = os.path.join(mw_building_volume, f"buildings_window{w}.tif")
    with rasterio.open(raster_path) as src:
        data = src.read(1)  # read first band

    im = ax.imshow(data, cmap='viridis')
    ax.set_title(f"Window {w}x{w}")
    ax.axis("off")

fig.suptitle("Building volume - Distance-weighted moving window", fontsize=16)
plt.tight_layout()
plt.show()



#### building height

In [ ]:
with rasterio.open(build_height_30m) as src:
    data = src.read(1).astype(np.float32)
    profile = src.profile

    # Loop over odd window sizes only
    for window_size in range(1, 101, 2):
        if window_size == 1:
            smoothed = data.copy()  # window 1 = original
        else:
            kernel = distance_weighted_kernel(window_size)
            smoothed = moving_window_weighted(data, kernel)

        # Save raster
        out_profile = profile.copy()
        out_profile.update(dtype=rasterio.float32, count=1, compress='lzw')
        out_path = os.path.join(
            mw_building_height, 
            f"building_height_window{window_size}.tif"
        )

        with rasterio.open(out_path, "w", **out_profile) as dst:
            dst.write(smoothed.astype(np.float32), 1)


In [ ]:
# plot

window_sizes = list(range(1, 21, 2))  # odd windows only 

fig, axes = plt.subplots(2, 5, figsize=(20, 8)) 
axes = axes.flatten()

for ax, w in zip(axes, window_sizes):
    raster_path = os.path.join(mw_building_height, f"building_height_window{w}.tif")
    with rasterio.open(raster_path) as src:
        data = src.read(1)  # read first band directly

    im = ax.imshow(data, cmap='viridis')
    ax.set_title(f"Window {w}x{w}")
    ax.axis("off")

fig.suptitle("Building height - Distance-weighted moving window", fontsize=16)
plt.tight_layout()
plt.show()

#### Tree volume

In [ ]:
with rasterio.open(tree_volume_30m) as src:
    data = src.read(1).astype(np.float32)
    profile = src.profile

    for window_size in range(1, 101, 2): # Loop over odd window sizes only
        if window_size == 1:
            smoothed = data.copy()  # window 1 = original
        else:
            kernel = distance_weighted_kernel(window_size)
            smoothed = moving_window_weighted(data, kernel)

        # Save raster
        out_profile = profile.copy()
        out_profile.update(dtype=rasterio.float32, count=1, compress='lzw')
        out_path = os.path.join(
            mw_tree_volume, 
            f"trees_window{window_size}.tif"
        )

        with rasterio.open(out_path, "w", **out_profile) as dst:
            dst.write(smoothed.astype(np.float32), 1)




In [ ]:
window_sizes = list(range(1, 21, 2))  # odd windows only

fig, axes = plt.subplots(2, 5, figsize=(20, 8))  
axes = axes.flatten()

for ax, w in zip(axes, window_sizes):
    raster_path = os.path.join(mw_tree_volume, f"trees_window{w}.tif")
    with rasterio.open(raster_path) as src:
        data = src.read(1)

    im = ax.imshow(data, cmap='YlGn', vmin=0, vmax=10000)
    ax.set_title(f"Window {w}x{w}")
    ax.axis("off")

fig.suptitle("Tree volume - Distance-weighted moving window", fontsize=16)
plt.tight_layout()
plt.show()


### Correlations with LST

In [ ]:
# general parameters
window_sizes = list(range(1, 101, 2))  # odd windows only

# Load LST
with rasterio.open(lst_clipped_path) as lst:
    lst_crs = lst.crs
    lst_crop = lst.read(1)          
    lst_transform = lst.transform  

#### Building area

In [ ]:
correlations = []

for w in window_sizes:
    raster_path = os.path.join(mw_building_area, f"building_area_window{w}.tif")
    with rasterio.open(raster_path) as src:
        raster_data = src.read(1).astype(float)  # read full raster

    # Pixel-by-pixel correlation using only valid LST pixels
    valid_mask = ~np.isnan(lst_crop) 
    var_vals = raster_data[valid_mask]
    lst_vals = lst_crop[valid_mask]

    corr = np.corrcoef(var_vals, lst_vals)[0, 1]
    correlations.append(corr)

# --- Plot ---
plt.figure(figsize=(8,5))
plt.plot(window_sizes, correlations, marker='o')
plt.xlabel("Window size (pixels)")
plt.ylabel("Correlation building area – LST")
plt.title("Correlation vs window size")
plt.grid(True)
plt.show()

#### Building volume

In [ ]:
correlations = []

for w in window_sizes:
    raster_path = os.path.join(mw_building_volume, f"buildings_window{w}.tif")
    with rasterio.open(raster_path) as src:
        raster_data = src.read(1).astype(float)

    valid_mask = ~np.isnan(lst_crop) 
    var_vals = raster_data[valid_mask]
    lst_vals = lst_crop[valid_mask]

    corr = np.corrcoef(var_vals, lst_vals)[0, 1]
    correlations.append(corr)

# Plot
plt.figure(figsize=(8,5))
plt.plot(window_sizes, correlations, marker='o')
plt.xlabel("Window size (pixels)")
plt.ylabel("Correlation building volume – LST")
plt.title("Correlation vs window size")
plt.grid(True)
plt.show()


#### Tree volume

In [ ]:
correlations = []

for w in window_sizes:
    raster_path = os.path.join(mw_tree_volume, f"trees_window{w}.tif")
    with rasterio.open(raster_path) as src:
        raster_data = src.read(1).astype(float)  

    valid_mask = ~np.isnan(lst_crop) 
    var_vals = raster_data[valid_mask]
    lst_vals = lst_crop[valid_mask]

    corr = np.corrcoef(var_vals, lst_vals)[0,1]
    correlations.append(corr)

# Plot
plt.figure(figsize=(8,5))
plt.plot(window_sizes, correlations, marker='o')
plt.xlabel("Window size (pixels)")
plt.ylabel("Correlation tree volume – LST")
plt.title("Correlation vs window size")
plt.grid(True)
plt.show()

#### building height

In [ ]:
correlations = []

for w in window_sizes:
    raster_path = os.path.join(mw_building_height, f"building_height_window{w}.tif")
    with rasterio.open(raster_path) as src:
        raster_data = src.read(1).astype(float)  

    valid_mask = ~np.isnan(lst_crop) 
    var_vals = raster_data[valid_mask]
    lst_vals = lst_crop[valid_mask]

    corr = np.corrcoef(var_vals, lst_vals)[0,1]
    correlations.append(corr)

# Plot
plt.figure(figsize=(8,5))
plt.plot(window_sizes, correlations, marker='o')
plt.xlabel("Window size (pixels)")
plt.ylabel("Correlation building height – LST")
plt.title("Correlation vs window size")
plt.grid(True)
plt.show()
